# Regresión Logística - Clasificación de Sistema Operativo

En esta notebook vamos a predecir qué sistema operativo utiliza un usuario
en base a su comportamiento en un sitio web.

**Características de entrada:**
- `duracion_segundos`: Duración de la visita en segundos
- `paginas_vistas`: Cantidad de páginas vistas durante la sesión
- `cantidad_acciones`: Cantidad de acciones del usuario
- `valor_acciones`: Suma del valor de las acciones

**Etiquetas de salida:**
- `0` → Windows
- `1` → Macintosh
- `2` → Linux

## 1. Importar librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns

print('Librerías importadas correctamente ✓')

## 2. Cargar y explorar los datos

In [ ]:
# Cargamos el dataset
df = pd.read_csv('usuarios_win_mac_lin.csv')

print('Primeras filas del dataset:')
df.head(10)

In [ ]:
# Información general del dataset
print('Forma del dataset:', df.shape)
print('\nInformación general:')
df.info()

In [ ]:
# Estadísticas descriptivas
print('Estadísticas descriptivas:')
df.describe()

In [ ]:
# Distribución de sistemas operativos
etiquetas = {0: 'Windows', 1: 'Macintosh', 2: 'Linux'}
conteo = df['sistema_operativo'].value_counts().rename(etiquetas)
print('Distribución de sistemas operativos:')
print(conteo)

## 3. Visualización exploratoria de los datos

In [ ]:
colores = {0: 'blue', 1: 'orange', 2: 'green'}
nombres_so = {0: 'Windows', 1: 'Macintosh', 2: 'Linux'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Distribución de datos por Sistema Operativo', fontsize=14, fontweight='bold')

# Duración vs Páginas Vistas
for so in [0, 1, 2]:
    subset = df[df['sistema_operativo'] == so]
    axes[0].scatter(subset['duracion_segundos'], subset['paginas_vistas'],
                    c=colores[so], label=nombres_so[so], s=80, edgecolors='black', linewidth=0.5)
axes[0].set_xlabel('Duración (segundos)')
axes[0].set_ylabel('Páginas Vistas')
axes[0].set_title('Duración vs Páginas Vistas')
axes[0].legend()

# Cantidad Acciones vs Valor Acciones
for so in [0, 1, 2]:
    subset = df[df['sistema_operativo'] == so]
    axes[1].scatter(subset['cantidad_acciones'], subset['valor_acciones'],
                    c=colores[so], label=nombres_so[so], s=80, edgecolors='black', linewidth=0.5)
axes[1].set_xlabel('Cantidad de Acciones')
axes[1].set_ylabel('Valor de Acciones')
axes[1].set_title('Acciones vs Valor')
axes[1].legend()

# Duración vs Cantidad Acciones
for so in [0, 1, 2]:
    subset = df[df['sistema_operativo'] == so]
    axes[2].scatter(subset['duracion_segundos'], subset['cantidad_acciones'],
                    c=colores[so], label=nombres_so[so], s=80, edgecolors='black', linewidth=0.5)
axes[2].set_xlabel('Duración (segundos)')
axes[2].set_ylabel('Cantidad de Acciones')
axes[2].set_title('Duración vs Cantidad Acciones')
axes[2].legend()

plt.tight_layout()
plt.show()

## 4. Preparar los datos para el modelo

In [ ]:
# Separamos las características (X) de las etiquetas (Y)
X = df[['duracion_segundos', 'paginas_vistas', 'cantidad_acciones', 'valor_acciones']].values
Y = df['sistema_operativo'].values

print('Forma de X (características):', X.shape)
print('Forma de Y (etiquetas):', Y.shape)
print('\nPrimeros 5 valores de X:')
print(X[:5])
print('\nPrimeros 5 valores de Y:')
print(Y[:5])

In [ ]:
# Dividimos en datos de entrenamiento (80%) y de prueba (20%)
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

print('Datos de entrenamiento:', X_train.shape)
print('Datos de prueba:       ', X_test.shape)

## 5. Crear y entrenar el modelo

In [ ]:
# Creamos el clasificador de Regresión Logística
clasificador = LogisticRegression(max_iter=1000, random_state=42)

# Entrenamos el modelo con los datos de entrenamiento
clasificador.fit(X_train, Y_train)

print('Modelo entrenado correctamente ✓')

## 6. Evaluación del modelo

In [ ]:
# Predicción sobre los datos de prueba
Y_pred = clasificador.predict(X_test)

print('Valores reales:    ', Y_test)
print('Valores predichos: ', Y_pred)

In [ ]:
# Accuracy (precisión global)
accuracy = accuracy_score(Y_test, Y_pred)
print(f'Accuracy del modelo: {accuracy * 100:.2f}%')

In [ ]:
# Reporte de clasificación completo
print('Reporte de Clasificación:')
print('=' * 50)
print(classification_report(Y_test, Y_pred,
                             target_names=['Windows', 'Macintosh', 'Linux']))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(Y_test, Y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Windows', 'Macintosh', 'Linux'],
            yticklabels=['Windows', 'Macintosh', 'Linux'])
plt.title('Matriz de Confusión', fontsize=13, fontweight='bold')
plt.ylabel('Valor Real')
plt.xlabel('Valor Predicho')
plt.tight_layout()
plt.show()

print('\nCómo leer la matriz:')
print('- Diagonal principal (↘): predicciones CORRECTAS')
print('- Fuera de la diagonal:   predicciones INCORRECTAS')

## 7. Predicción de nuevos usuarios

In [ ]:
# Definimos nuevos usuarios con sus características
nuevos_usuarios = np.array([
    [300, 7, 18, 52],   # Usuario A
    [750, 16, 11, 98],  # Usuario B
    [60,  2,  40, 22],  # Usuario C
])

# Realizamos la predicción
predicciones = clasificador.predict(nuevos_usuarios)
probabilidades = clasificador.predict_proba(nuevos_usuarios)

print('Predicciones para nuevos usuarios:')
print('=' * 55)
for i, (pred, proba) in enumerate(zip(predicciones, probabilidades)):
    so = nombres_so[pred]
    print(f'\nUsuario {chr(65+i)}: {nuevos_usuarios[i]}')
    print(f'  → Sistema Operativo predicho: {so} ({pred})')
    print(f'  → Probabilidades: Windows={proba[0]:.2%} | Mac={proba[1]:.2%} | Linux={proba[2]:.2%}')

## 8. Visualización de la zona de decisión (2 características)

Como solo podemos graficar en 2D, usamos las dos características más representativas: **duración** y **páginas vistas**.

In [ ]:
# Entrenamos un modelo auxiliar solo con 2 características para poder graficar
X_2d = df[['duracion_segundos', 'paginas_vistas']].values
Y_2d = df['sistema_operativo'].values

clasificador_2d = LogisticRegression(max_iter=1000, random_state=42)
clasificador_2d.fit(X_2d, Y_2d)

# Definimos el rango del gráfico
x_min, x_max = X_2d[:, 0].min() - 50, X_2d[:, 0].max() + 50
y_min, y_max = X_2d[:, 1].min() - 1,  X_2d[:, 1].max() + 1

# Creamos la malla de puntos
xx, yy = np.meshgrid(
    np.arange(x_min, x_max, 5),
    np.arange(y_min, y_max, 0.5)
)

# Predecimos el SO para cada punto de la malla
malla = clasificador_2d.predict(np.c_[xx.ravel(), yy.ravel()])
malla = malla.reshape(xx.shape)

# Graficamos
plt.figure(figsize=(10, 7))
plt.pcolormesh(xx, yy, malla, cmap=plt.cm.Pastel1, shading='auto')

# Puntos reales
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=Y_2d,
                      cmap=plt.cm.Set1, s=100,
                      edgecolors='black', linewidth=0.8)

# Leyenda
from matplotlib.patches import Patch
leyenda = [
    Patch(facecolor='#E41A1C', edgecolor='black', label='Windows (0)'),
    Patch(facecolor='#377EB8', edgecolor='black', label='Macintosh (1)'),
    Patch(facecolor='#4DAF4A', edgecolor='black', label='Linux (2)'),
]
plt.legend(handles=leyenda, loc='upper left', fontsize=11)

plt.title('Zonas de Decisión - Regresión Logística\n(Duración vs Páginas Vistas)',
          fontsize=13, fontweight='bold')
plt.xlabel('Duración de la Visita (segundos)', fontsize=11)
plt.ylabel('Páginas Vistas', fontsize=11)
plt.tight_layout()
plt.show()

print('El fondo de color muestra qué SO predice el modelo para cada zona del gráfico.')

## Conclusiones

- El modelo de **Regresión Logística** aprendió a distinguir entre los tres sistemas operativos en base al comportamiento del usuario en el sitio web.
- **Windows**: sesiones de duración media, páginas moderadas, acciones intermedias.
- **Macintosh**: sesiones largas, muchas páginas vistas, pocas acciones pero de alto valor.
- **Linux**: sesiones cortas, pocas páginas, pero gran cantidad de acciones rápidas.
- La **zona de decisión** muestra visualmente cómo el modelo separa las regiones del espacio de datos.